# PHOBERT MULTI-TASK (1 ENCODER + 2 HEADS: SENTIMENT & TOPIC)

In [1]:
#Notebook này chạy được trên cả Google Colab (GPU T4) và VS Code.
import os
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

In [2]:
IN_COLAB = 'google.colab' in sys.modules
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('in_colab:', IN_COLAB, '| device:', device)

in_colab: True | device: cuda


In [3]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
RAW_DIR = os.path.join(ROOT, 'data', 'raw')
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
MODEL_DIR = os.path.join(ROOT, 'models', 'phobert_multitask')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [4]:
#Nếu mở trực tiếp notebook này trên Google Colab mà chưa có train_clean.csv thì tự tải và làm sạch luôn
if not os.path.exists(os.path.join(PROC_DIR, 'train_clean.csv')):
    import re, unicodedata
    os.system('pip install -q gdown')
    import gdown
    urls = {
        'train_sentences.txt': '1nzak5OkrheRV1ltOGCXkT671bmjODLhP',
        'train_sentiments.txt': '1ye-gOZIBqXdKOoi_YxvpT6FeRNmViPPv',
        'train_topics.txt': '14MuDtwMnNOcr4z_8KdpxprjbwaQ7lJ_C',
        'dev_sentences.txt': '1sMJSR3oRfPc3fe1gK-V3W5F24tov_517',
        'dev_sentiments.txt': '1GiY1AOp41dLXIIkgES4422AuDwmbUseL',
        'dev_topics.txt': '1DwLgDEaFWQe8mOd7EpF-xqMEbDLfdT-W',
        'test_sentences.txt': '1aNMOeZZbNwSRkjyCWAGtNCMa3YrshR-n',
        'test_sentiments.txt': '1vkQS5gI0is4ACU58-AbWusnemw7KZNfO',
        'test_topics.txt': '1_ArMpDguVsbUGl-xSMkTF_p5KpZrmpSB'
    }
    for fname, fid in urls.items():
        out_p = os.path.join(RAW_DIR, fname)
        if not os.path.exists(out_p):
            gdown.download(id=fid, output=out_p, quiet=True)
    teencode_dict = {'k':'không','ko':'không','hok':'không','hong':'không','dc':'được','đc':'được','gv':'giảng viên','sv':'sinh viên','vs':'với','cx':'cũng','r':'rồi','bt':'bình thường','j':'gì','z':'vậy','v':'vậy'}
    compound_words = ['giảng viên','sinh viên','thầy cô','bài tập','kiến thức','thực hành','lý thuyết','phòng học','máy chiếu','điều hòa','máy lạnh','phòng máy','cơ sở vật chất','chương trình','giáo trình','tài liệu','nhiệt tình','dễ hiểu','khó hiểu']
    def quick_clean(t):
        t = unicodedata.normalize('NFC', str(t)).lower()
        t = re.sub(r'(.)\1{2,}', r'\1\1', t)
        t = re.sub(r'\b\w+\b', lambda m: teencode_dict.get(m.group(0), m.group(0)), t)
        for cw in compound_words:
            t = re.sub(r'\b' + cw + r'\b', cw.replace(' ', '_'), t)
        return re.sub(r'\s+', ' ', t).strip()
    for sp in ['train', 'dev', 'test']:
        s = open(os.path.join(RAW_DIR, f'{sp}_sentences.txt'), encoding='utf-8').read().splitlines()
        y1 = [int(x) for x in open(os.path.join(RAW_DIR, f'{sp}_sentiments.txt'), encoding='utf-8').read().splitlines()]
        y2 = [int(x) for x in open(os.path.join(RAW_DIR, f'{sp}_topics.txt'), encoding='utf-8').read().splitlines()]
        d = pd.DataFrame({'sentence': s, 'sentiment': y1, 'topic': y2})
        d['sentence_clean'] = d['sentence'].apply(quick_clean)
        d['sentence_seg'] = d['sentence_clean']
        d.to_csv(os.path.join(PROC_DIR, f'{sp}_clean.csv'), index=False, encoding='utf-8-sig')

In [5]:
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
df_test = pd.read_csv(os.path.join(PROC_DIR, 'test_clean.csv'))
for df in [df_train, df_dev, df_test]:
    df['sentence_seg'] = df['sentence_seg'].fillna('')

In [6]:
df_train.shape, df_dev.shape, df_test.shape

((11426, 7), (1583, 7), (3166, 7))

## 1. TOKENIZER VÀ DATASET

In [7]:
MODEL_NAME = 'vinai/phobert-base-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [8]:
class VSFCData(Dataset):
    def __init__(self, texts, sents, topics, tokenizer, max_len=128):
        self.texts = list(texts)
        self.sents = list(sents)
        self.topics = list(topics)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'sentiment': torch.tensor(self.sents[idx], dtype=torch.long),
            'topic': torch.tensor(self.topics[idx], dtype=torch.long)
        }

In [9]:
train_ds = VSFCData(df_train['sentence_seg'], df_train['sentiment'], df_train['topic'], tokenizer)
dev_ds = VSFCData(df_dev['sentence_seg'], df_dev['sentiment'], df_dev['topic'], tokenizer)
test_ds = VSFCData(df_test['sentence_seg'], df_test['sentiment'], df_test['topic'], tokenizer)

## 2. KIẾN TRÚC PHOBERT MULTI-TASK VÀ LOSS CHỐNG OVERFIT

In [10]:
class PhoBertMultiTask(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.sent_head = nn.Linear(hidden, 3)
        self.topic_head = nn.Linear(hidden, 4)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = self.drop(out.last_hidden_state[:, 0, :])
        return self.sent_head(cls_vec), self.topic_head(cls_vec)

In [11]:
#Dùng căn bậc hai của tỷ lệ nghịch tần suất lớp trên tập TRAIN (sqrt class weights) để không bị overfit vào lớp neutral
sent_counts = df_train['sentiment'].value_counts().sort_index().values
sent_weights = torch.tensor(np.sqrt(len(df_train) / (3.0 * sent_counts)), dtype=torch.float).to(device)

top_counts = df_train['topic'].value_counts().sort_index().values
top_weights = torch.tensor(np.sqrt(len(df_train) / (4.0 * top_counts)), dtype=torch.float).to(device)

loss_sent_fn = nn.CrossEntropyLoss(weight=sent_weights, label_smoothing=0.05)
loss_top_fn = nn.CrossEntropyLoss(weight=top_weights, label_smoothing=0.05)

## 3. DÒ HYPERPARAMETER (LEARNING RATE & DROPOUT) TRÊN TẬP DEV

In [12]:
#Thử nghiệm các tổ hợp (lr, dropout, weight_decay) trên 2,000 mẫu train và chấm điểm trên tập dev để tìm bộ tham số tốt nhất
from sklearn.metrics import f1_score, accuracy_score, classification_report

hp_candidates = [
    {'lr': 1e-5, 'dropout': 0.2, 'weight_decay': 0.01},
    {'lr': 2e-5, 'dropout': 0.3, 'weight_decay': 0.02},
    {'lr': 3e-5, 'dropout': 0.3, 'weight_decay': 0.01},
]

sub_loader = DataLoader(Subset(train_ds, range(min(2000, len(train_ds)))), batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=64, shuffle=False)

best_hp = hp_candidates[1]
best_search_score = -1.0

if torch.cuda.is_available():
    for hp in hp_candidates:
        torch.manual_seed(42)
        m_try = PhoBertMultiTask(dropout=hp['dropout']).to(device)
        opt_try = torch.optim.AdamW(m_try.parameters(), lr=hp['lr'], weight_decay=hp['weight_decay'])
        m_try.train()
        for batch in sub_loader:
            opt_try.zero_grad()
            ls, lt = m_try(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = loss_sent_fn(ls, batch['sentiment'].to(device)) + loss_top_fn(lt, batch['topic'].to(device))
            loss.backward()
            opt_try.step()
        m_try.eval()
        ds_p, dt_p = [], []
        with torch.no_grad():
            for batch in dev_loader:
                ls, lt = m_try(batch['input_ids'].to(device), batch['attention_mask'].to(device))
                ds_p.extend(ls.argmax(dim=-1).cpu().numpy())
                dt_p.extend(lt.argmax(dim=-1).cpu().numpy())
        sc = (f1_score(df_dev['sentiment'], ds_p, average='macro') + f1_score(df_dev['topic'], dt_p, average='macro')) / 2.0
        print('try hp:', hp, '-> dev mean macro f1:', round(sc, 4))
        if sc > best_search_score:
            best_search_score = sc
            best_hp = hp
        del m_try
        torch.cuda.empty_cache()

print('best_hp chon duoc:', best_hp)

try hp: {'lr': 1e-05, 'dropout': 0.2, 'weight_decay': 0.01} -> dev mean macro f1: 0.6842
try hp: {'lr': 2e-05, 'dropout': 0.3, 'weight_decay': 0.02} -> dev mean macro f1: 0.7295
try hp: {'lr': 3e-05, 'dropout': 0.3, 'weight_decay': 0.01} -> dev mean macro f1: 0.7218
best_hp chon duoc: {'lr': 2e-05, 'dropout': 0.3, 'weight_decay': 0.02}


## 4. HUẤN LUYỆN CHÍNH THỨC VỚI BEST HYPERPARAMETERS

In [13]:
torch.manual_seed(42)
model = PhoBertMultiTask(dropout=best_hp['dropout']).to(device)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

EPOCHS = 4 if torch.cuda.is_available() else 1
optimizer = torch.optim.AdamW(model.parameters(), lr=best_hp['lr'], weight_decay=best_hp['weight_decay'])
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

In [14]:
best_dev_f1 = 0.0
patience = 2
no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    tr_s_preds, tr_s_true = [], []
    for batch in train_loader:
        optimizer.zero_grad()
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        y_s = batch['sentiment'].to(device)
        y_t = batch['topic'].to(device)
        logits_s, logits_t = model(ids, mask)
        loss = loss_sent_fn(logits_s, y_s) + loss_top_fn(logits_t, y_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        tr_s_preds.extend(logits_s.argmax(dim=-1).detach().cpu().numpy())
        tr_s_true.extend(y_s.detach().cpu().numpy())
    
    train_f1_s = f1_score(tr_s_true, tr_s_preds, average='macro')
    
    model.eval()
    dev_s_preds, dev_t_preds = [], []
    with torch.no_grad():
        for batch in dev_loader:
            logits_s, logits_t = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            dev_s_preds.extend(logits_s.argmax(dim=-1).cpu().numpy())
            dev_t_preds.extend(logits_t.argmax(dim=-1).cpu().numpy())
    f1_s = f1_score(df_dev['sentiment'], dev_s_preds, average='macro')
    f1_t = f1_score(df_dev['topic'], dev_t_preds, average='macro')
    mean_f1 = (f1_s + f1_t) / 2.0
    print('epoch:', epoch + 1, '| loss:', round(total_loss / len(train_loader), 4), '| train_sent_f1:', round(train_f1_s, 4), '| dev_sent_f1:', round(f1_s, 4), '| dev_topic_f1:', round(f1_t, 4))
    if mean_f1 > best_dev_f1:
        best_dev_f1 = mean_f1
        no_improve = 0
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'phobert_multitask.pt'))
        tokenizer.save_pretrained(MODEL_DIR)
    else:
        no_improve += 1
        if no_improve >= patience:
            print('Early stopping tai epoch', epoch + 1)
            break

epoch: 1 | loss: 1.2418 | train_sent_f1: 0.6812 | dev_sent_f1: 0.7685 | dev_topic_f1: 0.7492
epoch: 2 | loss: 0.8145 | train_sent_f1: 0.8124 | dev_sent_f1: 0.7941 | dev_topic_f1: 0.7786
epoch: 3 | loss: 0.6692 | train_sent_f1: 0.8645 | dev_sent_f1: 0.8064 | dev_topic_f1: 0.7891
epoch: 4 | loss: 0.5784 | train_sent_f1: 0.8973 | dev_sent_f1: 0.8092 | dev_topic_f1: 0.7915


## 5. ĐÁNH GIÁ TRÊN TẬP TEST VÀ XUẤT FILE CHO NOTEBOOK 05 & APP

In [15]:
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'phobert_multitask.pt'), map_location=device))
model.eval()

test_s_preds, test_t_preds = [], []
test_s_probs, test_t_probs = [], []
with torch.no_grad():
    for batch in test_loader:
        logits_s, logits_t = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        prob_s = torch.softmax(logits_s, dim=-1).cpu().numpy()
        prob_t = torch.softmax(logits_t, dim=-1).cpu().numpy()
        test_s_preds.extend(prob_s.argmax(axis=-1))
        test_t_preds.extend(prob_t.argmax(axis=-1))
        test_s_probs.extend(prob_s.max(axis=-1))
        test_t_probs.extend(prob_t.max(axis=-1))

In [16]:
print('phobert test sentiment macro f1: ', f1_score(df_test['sentiment'], test_s_preds, average='macro'))
print('phobert test sentiment acc: ', accuracy_score(df_test['sentiment'], test_s_preds))
print('phobert test topic macro f1: ', f1_score(df_test['topic'], test_t_preds, average='macro'))
print('phobert test topic acc: ', accuracy_score(df_test['topic'], test_t_preds))

phobert test sentiment macro f1:  0.7928269990924307
phobert test sentiment acc:  0.9166140240050537
phobert test topic macro f1:  0.7774654305898754
phobert test topic acc:  0.8651295009475679


In [17]:
print(classification_report(df_test['sentiment'], test_s_preds, target_names=['negative', 'neutral', 'positive']))
print(classification_report(df_test['topic'], test_t_preds, target_names=['lecturer', 'training_program', 'facility', 'others']))

              precision    recall  f1-score   support

    negative       0.93      0.94      0.94      1409
     neutral       0.53      0.48      0.50       167
    positive       0.94      0.94      0.94      1590

    accuracy                           0.92      3166
   macro avg       0.80      0.79      0.79      3166
weighted avg       0.91      0.92      0.92      3166

                  precision    recall  f1-score   support

        lecturer       0.96      0.89      0.92      2290
training_program       0.69      0.81      0.74       572
        facility       0.89      0.94      0.91       145
          others       0.48      0.59      0.53       159

        accuracy                           0.87      3166
       macro avg       0.75      0.81      0.78      3166
    weighted avg       0.88      0.87      0.87      3166



In [18]:
df_pred = df_test.copy()
df_pred['pred_sentiment'] = test_s_preds
df_pred['pred_topic'] = test_t_preds
df_pred['sentiment_conf'] = np.round(test_s_probs, 4)
df_pred['topic_conf'] = np.round(test_t_probs, 4)
df_pred['needs_review'] = (df_pred['sentiment_conf'] < 0.65) | (df_pred['topic_conf'] < 0.65)
df_pred.to_csv(os.path.join(PROC_DIR, 'test_pred_phobert.csv'), index=False, encoding='utf-8-sig')

In [19]:
if IN_COLAB:
    import zipfile
    zip_path = 'phobert_outputs.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(os.path.join(PROC_DIR, 'test_pred_phobert.csv'), 'data/processed/test_pred_phobert.csv')
        for root_d, _, files_d in os.walk(MODEL_DIR):
            for f in files_d:
                full_p = os.path.join(root_d, f)
                rel_p = os.path.relpath(full_p, ROOT)
                zf.write(full_p, rel_p)
    from google.colab import files
    files.download(zip_path)